In [ ]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")

In [ ]:
experiment_file = '../../experiments/parallelized_experiments/output/cosine_curve_amplitude_full_period/2023_12_17_15_52//experiment_result.json'

stiffness_path = '../../experiments/parallelized_experiments/output/cosine_curve_amplitude_full_period/2023_12_17_15_52/'

### Overview

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [ ]:
df = pd.DataFrame(data['data'])
fig, axes = plt.subplots(nrows = 1, ncols = 3, figsize = (20, 5))
a = (df.hist('Ipu simulation succeed', ax = axes[0]), df.hist('Planar equilibrium', ax = axes[1]), df.hist('Simulation Kappa value', ax = axes[2]))

In [ ]:
import visualize_stiffness
import importlib
importlib.reload(visualize_stiffness)

In [ ]:
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [ ]:
kappa_path = None

In [ ]:
name = 'cosine_curve_amplitude_full_period'

In [ ]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, name, valid_tags, plot_data = False)

In [ ]:
parameters = (np.array(data['pattern_parameters'][0]['values']))

In [ ]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [ ]:
parameters[np.argmax(min_bending_stiffness)]

In [ ]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [ ]:
min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

In [ ]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, name, valid_tags)

### Get scale function convex hull

In [ ]:
import matplotlib.cm as cm
import matplotlib as mpl

In [ ]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

In [ ]:
hull

### Validate the max and min scale factors are aligned with the x and y axis

In [ ]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [ ]:
eqns = hull.equations

In [ ]:
hull.max_bound, hull.min_bound

In [ ]:
import parametrization_helper, importlib
importlib.reload(parametrization_helper)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, max_scale_factors, min_scale_factors)

### Generate data without augmenting

In [ ]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [ ]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [ ]:
# for i in range(5):
#     for j in range(30):
#         stiffness_coefficients[:, i] = parametrization_helper.savitzky_golay(stiffness_coefficients[:, i], 11, 3) # window size 51, polynomial order 3

In [ ]:
np.set_printoptions(suppress=True, precision=4)

In [ ]:
np.argmax(stiffness_coefficients[:, 1]), np.argmax(stiffness_coefficients[:, 2])

In [ ]:
def get_stiffness_polynomial(s, theta):
    return s[0] * np.cos(theta)**2 * np.sin(theta)**2 + s[1] * np.cos(theta)**3 * np.sin(theta) + s[2] * np.cos(theta) * np.sin(theta)**3 + s[3] * np.cos(theta)**4 + s[4] * np.sin(theta)**4

In [ ]:
grid_data = np.zeros((9, len(parameters)))

In [ ]:
for i in range(len(parameters)):
    grid_data[0][i] = max_scale_factors[i]
    grid_data[1][i] = min_scale_factors[i]
    grid_data[2][i] = x_scale_factors[i]
    grid_data[3][i] = y_scale_factors[i]
    for s in range(5):
        grid_data[4 + s][i] = stiffness_coefficients[i][s]

In [ ]:
# np.save("grid_pattern_1.npy", grid_pattern_1)
# np.save("grid_pattern_2.npy", grid_pattern_2)
# np.save("grid_data.npy", grid_data)

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, (parameters))

In [ ]:
grid_data.shape

In [ ]:
scale_factors_grid_data = np.zeros((2, len(parameters)))
for i in range(len(parameters)):
    scale_factors_grid_data[0][i] = x_scale_factors[i]
    scale_factors_grid_data[1][i] = y_scale_factors[i]
scale_factors_splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(scale_factors_grid_data, (parameters))

In [ ]:
test_parameters = np.linspace(0, 0.9, 100)

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
# titles = ['max scale factors', 'min scale factors', 's1', 's2', 's3', 's4', 's5']

for i in range(9):
    axes[i].plot(test_parameters, splines[i * 3 + 1](test_parameters))
    axes[i].set_title(titles[i], fontsize=21)

In [ ]:
stiffness_coefficients = np.array(stiffness_coefficients)

In [ ]:
stiffness_coefficients.shape

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
data = [max_scale_factors, min_scale_factors, x_scale_factors, y_scale_factors, stiffness_coefficients[:, 0], stiffness_coefficients[:, 1], stiffness_coefficients[:, 2], stiffness_coefficients[:, 3], stiffness_coefficients[:, 4]]

for i in range(9):
    axes[i].plot(parameters, data[i])
    axes[i].set_title(titles[i], fontsize=21)

### End data generating

### Parametrization

In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
import utils, mesh_utilities
importlib.reload(utils)

In [ ]:
target_surf = mesh.Mesh("../../../../examples/igloo.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
lines = np.array(eqns)

### New local global with convex hull

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.setLines(eqns)

lg.alphaMin = hull.min_bound[0]
lg.alphaMax = hull.max_bound[0]

lg.betaMin = hull.min_bound[1]
lg.betaMax = hull.max_bound[1]

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
lg.alphaMin, lg.alphaMax, lg.betaMin, lg.betaMax

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(lg, show_main = True)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, lg.getAlphas(), lg.getBetas())

### Pattern parameters optimization

In [ ]:
default_pattern_params = [0.3]  * len(lg.getAlphas())

In [ ]:
mat_info = np.array(default_pattern_params).reshape((1, len(lg.getAlphas())))

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[0.01, 0.3]])
rparam.diffRegW = 0.0

In [ ]:
visualization.visualize_both(rparam, height = 4)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [ ]:
rparam.bendRegW = 1

In [ ]:
rparam.energy(PET.RGP)

In [ ]:
rparam.energy(PET.Bending)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.RGP, PET.Bending]))

In [ ]:
def optimize_rparam(param, patternRegW, phiRegW, bendRegW = 0.0, update_uv = True, niter = 100):
    param.patternRegW = patternRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = niter
    opts.gradTol = 1e-9
    opts.factorizer = opts.factorizer.CatamariNesdis
    benchmark.reset()
    
    if update_uv:
        fixedvars = [param.uOffset(), param.vOffset(), param.phiOffset()]
    else:
        fixedvars = range(param.stretchOffset())

    cr = parametrization.pattern_parametrization_knitro(param, opts.niter, fixedvars)
    benchmark.report()
    return cr

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 20, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, rparam.getAlphas(), rparam.getBetas())

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=1)

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False, width = 5, height = 5)

In [ ]:
visualization.visualizeChannelOrientationWithIsotropicPoints(rparam, quiver=visualization.QuiverVisualization.PER_TRI, orientationHue=False, width = 5, height = 5, use_x_axis=True)

In [ ]:
importlib.reload(visualization)
importlib.reload(parametrization_helper)

In [ ]:
rparam.get_stretch_angle_offset_from_pattern_params(rparam.getMatInfoArgs())

In [ ]:
max(parametrization_helper.get_stretch_angle_offset_from_pattern_params(scale_factors_splines, rparam.getPatternParams().reshape(1, -1)))

In [ ]:
rparam.get_stretch_angle_offset_from_pattern_params(rparam.getPatternParams().reshape(1, -1))

## Upsampling and channel generation

In [ ]:
nsubdiv=3
upsampledMesh, upsampledAngles, upsampledPatternParams = rparam.upsampledVertexLeftStretchAnglesAndPatternParameters(nsubdiv)
upsampleMesh_vertices = upsampledMesh.vertices()
upsampleMesh_triangles = upsampledMesh.triangles()
angle_offset = parametrization_helper.get_stretch_angle_offset_from_pattern_params(scale_factors_splines, np.array(upsampledPatternParams).reshape((1, -1)))
upsampledAngles += angle_offset

In [ ]:
# min(angle_offset), max(angle_offset)

In [ ]:
np.save("upsampleMesh_vertices.npy", upsampledMesh.vertices())
np.save("upsampleMesh_triangles.npy", upsampledMesh.triangles())
np.save("upsampleAngles.npy", upsampledAngles)
np.save("upsampledPatternParams.npy", upsampledPatternParams)

In [ ]:
upsampleMesh_vertices = np.load("upsampleMesh_vertices.npy")
upsampleMesh_triangles = np.load("upsampleMesh_triangles.npy")
upsampleAngles = np.load("upsampleAngles.npy")
upsampledPatternParams = np.load("upsampledPatternParams.npy")

In [ ]:
np.set_printoptions(suppress=True)

In [ ]:
max(upsampledPatternParams[0])

In [ ]:
amp_data =  upsampledPatternParams[0]


In [ ]:
import igl

In [ ]:
from parametrization_helper import get_distance_to_line_segments

In [ ]:
def fusing_curve_polyline(patternParams):
#     Draw cosine curves.
    amp = patternParams[0]
    def get_y_from_x(x):
        return amp * np.cos(x) * 0.5 * np.pi + np.pi / 2
    
    x_coords = np.linspace(-np.pi, np.pi, 20)
    y_coords = get_y_from_x(x_coords)
    x_coords += np.pi
    x_coords /= 2
    polyline = np.concatenate(((y_coords).reshape(-1, 1), (x_coords).reshape(-1, 1)), axis = 1)
    return polyline

In [ ]:
def pattern_function(theta, gamma, patternParams, margin, draw_boundary = False):
    if draw_boundary:
#         This is for debugging only and shouldn't be used for generating the inflatable mesh.
        if (theta < 0.1):
            return - margin
        if (gamma < 0.1):
            return - margin
    # Gamma is y, theta is x
    # Gamma theta are between 0 and pi
    polyline = fusing_curve_polyline(patternParams)    
    polyline_dist = get_distance_to_line_segments(np.array([theta, gamma]), polyline)
    return polyline_dist - margin

In [ ]:
import shapely

In [ ]:
def convert_shapely_objs_to_edge_soup(results):
    if isinstance(results, shapely.geometry.LineString):
        polyline = np.array(results.coords)
        polyline = np.concatenate((polyline, np.zeros((len(polyline), 1))), axis = 1)
        # edge soup should be a list of pairs of points
        edgeSoup = [polyline[i:i+2] for i in range(len(polyline)-1)]
    elif isinstance(results, shapely.geometry.MultiLineString):
        polylines = [np.array(line.coords) for line in results.geoms]
        edgeSoup = []
        for polyline in polylines:
            polyline = np.concatenate((polyline, np.zeros((len(polyline), 1))), axis = 1)
            edgeSoup.extend([polyline[i:i+2] for i in range(len(polyline)-1)])
    elif isinstance(results, shapely.geometry.Point):
        coords = np.array(results.coords)
        coords = np.concatenate((coords, np.zeros((len(coords), 1))), axis = 1)[0]
        edgeSoup = [[coords, coords]]
    elif isinstance(results, shapely.geometry.GeometryCollection):
        for obj in results.geoms:
            print(obj)
        edgeSoup = []
        for obj in results.geoms:
            edgeSoup.extend(convert_shapely_objs_to_edge_soup(obj))
    else:
        raise ValueError('Unexpected intersection result type: {}'.format(type(results)))
    return edgeSoup

In [ ]:
def compute_barycentric_coordinates(triangle, points):
    # Triangle vertices
    v0, v1, v2 = triangle[0], triangle[1], triangle[2]
    
    # Area of the triangle
    e1 = v1 - v0
    e2 = v2 - v0
    area = np.abs(0.5 * np.cross(e1, e2))
    if (area < 1e-8):
        return None
    
    # Initialize array for barycentric coordinates
    barycentric_coordinates = np.zeros((points.shape[0], 3))
    
    for i, p in enumerate(points):
        # Area of the triangle formed by p, v1 and v2
        e1 = v1 - p
        e2 = v2 - p
        area1 = np.abs(0.5 * np.cross(e1, e2))
        
        # Area of the triangle formed by p, v0 and v2
        e1 = v0 - p
        e2 = v2 - p
        area2 = np.abs(0.5 * np.cross(e1, e2))
        
        # Barycentric coordinates
        barycentric_coordinates[i, 0] = area1 / area
        barycentric_coordinates[i, 1] = area2 / area
        barycentric_coordinates[i, 2] = 1 - barycentric_coordinates[i, 0] - barycentric_coordinates[i, 1]
    
    return barycentric_coordinates

In [ ]:
barycentric_coords_soup_results = []
theta_gamma_results = []
edge_soup_results = []
patternParams_results = []
def pattern_polyline_function(theta_gamma, patternParams):
    default = [[[1, 0, 0], [0, 1, 0]], [[0, 1, 0], [0, 0, 1]], [[1, 0, 0], [0, 0, 1]]]
    default = []

    clip_tri = shapely.geometry.Polygon(theta_gamma)
    # print("distance to polyline: ", pattern_function(theta_gamma[0][0], theta_gamma[0][1], patternParams, 0.0), pattern_function(theta_gamma[1][0], theta_gamma[1][1], patternParams, 0.0), pattern_function(theta_gamma[2][0], theta_gamma[2][1], patternParams, 0.0))
    # print(theta_gamma, patternParams)
    polyline = fusing_curve_polyline(patternParams)
    # print("clip tri: ", theta_gamma, "polyline: ", polyline)
    line = shapely.geometry.LineString(polyline)
    if not line.intersects(clip_tri):
        barycentric_coords_soup_results.append(default)
        theta_gamma_results.append(theta_gamma)
        edge_soup_results.append([])
        patternParams_results.append(patternParams)
        # print("no intersection")
        return default
    # print("has intersection")
    results = line.intersection(clip_tri)
    edge_soup = np.array(convert_shapely_objs_to_edge_soup(results))
    flat_edge_soup = edge_soup.reshape((-1, 3))[:, :2]
    barycentric_coords_soup = compute_barycentric_coordinates(np.array(theta_gamma), flat_edge_soup)
    if (barycentric_coords_soup is None):
        barycentric_coords_soup_results.append(default)
        theta_gamma_results.append(theta_gamma)
        edge_soup_results.append(edge_soup)
        patternParams_results.append(patternParams)
        print("soup is empty")
        
        return default
    barycentric_coords_soup_results.append(default + list(barycentric_coords_soup.reshape((-1, 2, 3))))
    theta_gamma_results.append(theta_gamma)
    edge_soup_results.append(edge_soup)
    patternParams_results.append(patternParams)
    return default + list(barycentric_coords_soup.reshape((-1, 2, 3)))

In [ ]:
pattern_function(np.pi / 4, np.pi / 10, [0.], 0.)

In [ ]:
# upsampledPatternParams = np.ones_like(upsampledPatternParams) * 0.4

In [ ]:
import time
start_time = time.time()
(sdfVertices, sdfTris, sdf, edge_soup) = wall_generation.evaluate_cross_field_custom_pattern(upsampleMesh_vertices, upsampleMesh_triangles, upsampleAngles, upsampledPatternParams, pattern_function, pattern_polyline_function, frequency=0.05, margin = 0.0, nsubdiv = 0)
print(time.time() - start_time)

# pickle.dump((sdfVertices, sdfTris, sdf), open('stripe_sdf_ns4_f100.pkl', 'wb'))

# import pickle, mesh, wall_generation, visualization, numpy as np
# (sdfVertices, sdfTris, sdf) = pickle.load(open('stripe_sdf_ns4_f100.pkl', 'rb'))

In [ ]:
theta_gamma_results = np.array(theta_gamma_results)

In [ ]:
edge_soup_results[0], theta_gamma_results[1], barycentric_coords_soup_results[0]

In [ ]:
def visualize_barycentric(index):
    v0 = theta_gamma_results[index][0]
    v1 = theta_gamma_results[index][1]
    v2 = theta_gamma_results[index][2]
    

    fig, ax = plt.subplots(figsize=(12, 6))
    for edge in edge_soup_results[index]:
        plt.plot(edge[:, 0], edge[:, 1])
    
    plt.scatter(theta_gamma_results[index][:, 0], theta_gamma_results[index][:, 1])
    plt.plot(theta_gamma_results[index][:, 0], theta_gamma_results[index][:, 1])

    for bary in barycentric_coords_soup_results[index][:]:
        p1 = np.zeros(2)
        p2 = np.zeros(2)
        for i in range(3):
            p1 += theta_gamma_results[index][i] * bary[0][i]
            p2 += theta_gamma_results[index][i] * bary[1][i]
        
        barycentric_edge_soup = np.array([p1, p2])
        plt.plot(barycentric_edge_soup[:, 0], barycentric_edge_soup[:, 1], '--')

    polyline = fusing_curve_polyline(patternParams_results[0])
    plt.plot(polyline[:, 0], polyline[:, 1], '+')
    ax.set_xlim([0, np.pi])
    plt.xlim(0, np.pi)
    ax.axis('equal')


In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
interact(visualize_barycentric, index=widgets.IntSlider(min=0, max=len(barycentric_coords_soup_results) - 1, step=1, value=0));

In [ ]:
importlib.reload(visualization)

import matplotlib as mpl

In [ ]:
len(edge_soup)

In [ ]:
new_edge_soup = np.array(edge_soup)

In [ ]:
points = new_edge_soup.reshape((-1, 3))

In [ ]:
edges = [[2 * i, 2 * i + 1] for i in range(len(new_edge_soup))]

In [ ]:
old_mesh = MeshFEM.mesh.Mesh(upsampleMesh_vertices, upsampleMesh_triangles)
visualization.plot_2d_mesh(old_mesh)

In [ ]:
new_mesh = MeshFEM.mesh.Mesh(sdfVertices, sdfTris)

In [ ]:
len(sdfVertices), len(upsampleMesh_vertices)

In [ ]:
visualization.plot_2d_mesh(new_mesh, width = 5, height = 5)

In [ ]:
new_edges = list(sdfTris[:, 1:]) + list(sdfTris[:, :2]) + list(sdfTris[:, [0, 2]])

In [ ]:
# visualization.plot_line_segments(sdfVertices, new_edges, width = 5, height = 5)

In [ ]:
len(edges)

In [ ]:
visualization.plot_line_segments(points, edges, width = 50, height = 50)

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 5, height=5)

In [ ]:
importlib.reload(visualization)
visualization.scalarFieldPlotZeroContourFast(sdfVertices, sdfTris, sdf, width = 15, height=10, cmap = mpl.colormaps["PiYG"])

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=0.4,
                                              minContourLen=1)

visualization.plot_line_segments(pts, edges, width=15, height=15)

## Meshing and inflation simulation

In [ ]:
import sheet_meshing, inflation


In [ ]:
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=1e0)


In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(iwv) == 1)[0], width=10, height=10)


In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(iwv) != 0)

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [ ]:
isheet.pressure = 1e-2

In [ ]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [ ]:
import gzip

In [ ]:
pickle.dump(isheet,  gzip.open("igloo_pattern_optimized_2023_12_17_low_frequency_with_bending_high_resolution.pkl.gz", 'wb'))

### Generate Fabrication Files

In [ ]:
scaleFactor = 1 # Factor for fine-tuning size to fit the machine's build area
channelMargin = 0 / scaleFactor # 8mm channel margin
tabMargin = 2 / scaleFactor # 2mm tab margin

In [ ]:

import inflation

targetSurf = target_surf
iwv = [isheet.isWallVtx(i) for i in range(isheet.mesh().numVertices())]

In [ ]:
isheet.mesh()

In [ ]:
mesh = isheet.mesh()

In [ ]:
mesh.save("igloo_2D.obj")

In [ ]:
np.save("igloo_is_wall.npy", iwv)

In [ ]:
uv = rparam.uv()

In [ ]:
# !pip install shapely==1.7.0

In [ ]:
import shapely

In [ ]:
import fabrication

In [ ]:
importlib.reload(utils)
importlib.reload(fabrication)
importlib.reload(shapely)
import shapely.geometry as shp


In [ ]:
shapely.__version__

In [ ]:
# new_fabrication.writeFabricationData('fabrication_data/igloo/free_bdry', isheet.mesh(), iwv, targetSurf, uv,
#                                  scale=scaleFactor,
#                                  channelMargin=channelMargin, fuseSeamWidth=None,
#                                  overlap=0.0, smartOuterChannel=True)

In [ ]:
import fabrication
fabrication.writeFabricationData('fabrication_data/igloo_2023_12_17/free_bdry_low_frequency', isheet.mesh(), isheet.mesh(), iwv, targetSurf, uv,
                                 scale=scaleFactor, numTabs=0, inletOffset=0, tabOffset=0.60 / 80,
                                 channelMargin=channelMargin, tabMargin=tabMargin, tabWidth=5, tabHeight=8, fuseSeamWidth=None, inletScale=None,
                                 overlap=0.0, smartOuterChannel=False)

In [ ]:
# import fabrication
# fabrication.writeFabricationData('fabrication_data/igloo/free_bdry', isheet.mesh(), isheet.mesh(), iwv, targetSurf, uv,
#                                  scale=scaleFactor, numTabs=0, inletOffset=0.742, tabOffset=0.60 / 80,
#                                  channelMargin=channelMargin, tabMargin=tabMargin, tabWidth=5, tabHeight=8, fuseSeamWidth=1.0, inletScale=12 / channelMargin / scaleFactor,
#                                  overlap=0.0, smartOuterChannel=True)

In [ ]:
from IPython.display import SVG, display


In [ ]:
display(SVG(filename = "fabrication_data/igloo/free_bdry_low_frequency/orig.wall_boundaries.svg"))